<a href="https://colab.research.google.com/github/eliabrodsky/la_data/blob/main/Hospital_Analysis_VBC_EB_Sept_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Louisiana Rural Hospital Sustainability, VBC and Shared EHR

Segments Louisiana's 64 rural hospitals on financial position, assesses which can
benefit from entering value-based payment arrangements, and identifies which would
gain most from a shared Epic instance.

**Data:** CMS Hospital Provider Cost Reports (HCRIS, form CMS-2552-10), FY2021–FY2023,
with the LIHNC roster and the Epic state instance pipeline.

---
# 1 &nbsp; Logic of the analysis

## The starting point

LIHNC exists. Twenty-four hospitals joined it. Separately, a set of hospitals has
engaged with the state Epic instance. Both groups formed through internal factors we
cannot observe directly: relationships, board decisions, existing referral ties, who
knew whom.

The premise of this analysis is that financial position drives these decisions.
A hospital under pressure has more to gain from a network, and a hospital with capital
has more ability to fund an EHR build. If that premise holds, the financial data will
sort hospitals into groups that line up with LIHNC membership and Epic interest.

## What the analysis tests

**Do financial groups exist?** Cluster the hospitals on cost report data and test
whether the resulting groups survive resampling.

**Do those groups explain LIHNC and Epic?** Predict membership in each program from
financial features alone, and compare against the base rate.

**If not, what does?** Screen the non-financial variables available, and specify what
would be needed to answer the question properly.

## What drives a rural hospital's financial position

Three things, in order of magnitude.

**Reimbursement class.** A Critical Access Hospital is paid close to cost. An IPPS
hospital receives a fixed price per case. That difference sets the shape of the income
statement before any management decision is made.

**Ownership structure.** A parish hospital district holds its own cash. A hospital
inside a corporate system has cash swept to the parent. The balance sheet reports the
treasury arrangement.

**Operating performance.** What remains once the first two are accounted for.

Value-based care participation depends on a fourth thing, independent of these:
control of primary care billing. Medicare attributes a patient to whoever bills their
primary care visits, so a hospital with no employed primary care providers brings no
attributable lives, whatever its size or solvency.

## How the analysis proceeds

| Section | Purpose |
|---|---|
| **2 Load** | Build a three-year panel and derive comparable ratios |
| **3 Explore** | Establish which metrics are comparable across hospitals |
| **4 Cluster** | Find financial groups and test whether they hold |
| **5 Interpret** | Test the groups against LIHNC and Epic membership |
| **6 Explain** | Screen alternative explanations and score shared-EHR benefit |
| **7 Place** | Test whether rurality and population explain the grouping |

## Two properties of the data that shape the method

**Position moves year to year.** Segments assigned from a single year hold across three
years for about half of these hospitals. Cost-settled hospitals swing with settlement
timing. Features are built as level, slope and volatility across all three years.

**Structure contaminates the balance sheet.** Reimbursement class and ownership enter as
stratification variables. Comparisons are made within them, and they stay out of the
clustering inputs.

## Definitions

| Term | Meaning here |
|---|---|
| **Rural hospital** | The 64-hospital universe with a Medicare cost report, from the CMS CAH list and the LDH rural designation lists |
| **Reimbursement class** | Medicare class: CAH, IPPS, SCH, RRC, REH |
| **Ownership stratum** | Type of Control from Worksheet S-2, collapsed to Government, Nonprofit, Proprietary |
| **VBC potential** | Risk-bearing capacity, operating performance, and attributable primary care lives. All three required |
| **Shared EHR benefit** | Implementation cost relative to the hospital's revenue and its clinical volume |

---
# 2 &nbsp; Loading the data

## 2.1 &nbsp; Environment

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal, linregress
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import cdist

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

warnings.filterwarnings('ignore', category=FutureWarning)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Environment ready.')

## 2.2 &nbsp; Source files

Three annual CMS cost report public use files, read directly from GitHub.

`rural_ccn.csv` defines the 64-hospital rural universe and carries the LIHNC and Epic
pipeline flags. It also maps filing names to working names, since hospitals file under
legal entities: Prevost is West Ascension, Richland Parish Hospital Service District is
Richardson, Riverland is Trinity.

In [ ]:
BASE = 'https://raw.githubusercontent.com/eliabrodsky/la_data/main/'

FILES = {
    2021: BASE + 'CostReport_2021_Final.csv',
    2022: BASE + 'CostReport_2022_Final.csv',
    2023: BASE + 'CostReport_2023_Final.csv',
}

CROSSWALK = BASE + 'rural_ccn.csv'

probe = pd.read_table(FILES[2023], sep=',', header=0, nrows=5, low_memory=False)
print(f'Source reachable. {probe.shape[1]} columns per annual file.')
probe.head()

## 2.3 &nbsp; Filter to Louisiana and derive the ratio set

Per-day ratios divide by the actual number of days in the reporting period. Cost reports
do not all cover a full year. Six Louisiana reports cover under 330 days and one covers
32 days, where an annual denominator would report 5,140 days of cash on hand in place
of 442.

In [ ]:
CONTROL = {
    1: 'Nonprofit-Church',       2: 'Nonprofit-Other',
    3: 'Proprietary-Individual', 4: 'Proprietary-Corp',
    5: 'Proprietary-Partnership', 6: 'Proprietary-Other',
    7: 'Gov-Federal',            8: 'Gov-City-County',
    9: 'Gov-County',            10: 'Gov-State',
    11: 'Gov-Hospital District', 12: 'Gov-City',
    13: 'Gov-Other',
}


def load_year(year, url):
    """Read one annual cost report file, keep Louisiana, derive the ratio set."""
    d = pd.read_table(url, sep=',', header=0, low_memory=False)
    d = d[d['State Code'] == 'LA'].copy()

    def col(name):
        if name not in d.columns:
            return pd.Series(np.nan, index=d.index)
        return pd.to_numeric(d[name], errors='coerce')

    begin  = pd.to_datetime(d['Fiscal Year Begin Date'], errors='coerce')
    end    = pd.to_datetime(d['Fiscal Year End Date'],   errors='coerce')
    period = (end - begin).dt.days.clip(lower=1)

    cash = col('Cash on Hand and in Banks').fillna(0) + col('Temporary Investments').fillna(0)
    opex = col('Less Total Operating Expense')
    dep  = col('Depreciation Cost').fillna(0)
    npr  = col('Net Patient Revenue')
    oth  = col('Total Other Income').fillna(0)

    out = pd.DataFrame({
        'fy':          year,
        'ccn':         d['Provider CCN'].astype(str).str.zfill(6),
        'hospital':    d['Hospital Name'].str.strip(),
        'city':        d['City'].str.strip().str.title(),
        'hcris_class': d['CCN Facility Type'],
        'control':     pd.to_numeric(d['Type of Control'], errors='coerce').map(CONTROL),
        'fy_days':     period,
        'npr':         npr,
        'beds':        col('Number of Beds'),
        'fte':         col('FTE - Employees on Payroll'),
    })

    # Liquidity: cash relative to daily cash operating expense, depreciation removed
    out['days_cash']     = (cash / ((opex - dep) / period)).where((opex - dep) > 0)

    # Capital structure
    out['equity_ratio']  = (col('Total Fund Balances') / col('Total Assets')
                            ).where(col('Total Assets') > 0)
    out['current_ratio'] = (col('Total Current Assets') / col('Total Current Liabilities')
                            ).where(col('Total Current Liabilities') > 0)

    # Profitability: on patient care alone, and on the bottom line
    out['pt_svc_margin'] = (col('Net Income from Service to Patients') / npr).where(npr > 0)
    out['total_margin']  = (col('Net Income') / (npr + oth)).where((npr + oth) > 0)

    print(f'  FY{year}: {len(out)} Louisiana hospitals')
    return out


print('Loading annual cost report files')
panel = pd.concat([load_year(y, u) for y, u in FILES.items()], ignore_index=True)

## 2.4 &nbsp; Restrict to the rural universe and assign ownership stratum

In [ ]:
xwalk = pd.read_table(CROSSWALK, sep=',', header=0, dtype=str)
xwalk['ccn'] = xwalk['ccn'].str.zfill(6)

panel = panel[panel['ccn'].isin(set(xwalk['ccn']))].copy()

panel['stratum'] = panel['control'].map(
    lambda c: 'Government'  if str(c).startswith('Gov')
    else     ('Proprietary' if str(c).startswith('Proprietary')
    else      'Nonprofit')
)

print(f"{len(panel)} hospital-years  |  {panel['ccn'].nunique()} hospitals  "
      f"|  FY{panel['fy'].min()}-FY{panel['fy'].max()}")
print()
print('Ownership stratum, most recent year')
print(panel[panel['fy'] == panel['fy'].max()]['stratum'].value_counts().to_string())

---
# 3 &nbsp; Exploratory analysis

Establishes which metrics are comparable across hospitals, which determines what the
clustering model may contain.

## 3.1 &nbsp; Distribution by ownership

In [ ]:
latest = panel[panel['fy'] == panel['fy'].max()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, title in zip(
        axes,
        ['days_cash', 'pt_svc_margin'],
        ['Days cash on hand', 'Patient services margin']):

    order = ['Government', 'Nonprofit', 'Proprietary']
    data = [latest.loc[latest['stratum'] == s, metric].dropna() for s in order]
    ax.boxplot(data, tick_labels=order, showfliers=False)
    ax.set_title(title)
    ax.axhline(0, color='#c3c2b7', linewidth=0.8)

axes[0].set_ylabel('Days')
axes[1].set_ylabel('Share of net patient revenue')
plt.tight_layout()
plt.show()

print(latest.groupby('control')[['days_cash', 'equity_ratio', 'total_margin']]
      .agg(['size', 'median']).round(3)
      .sort_values(('days_cash', 'size'), ascending=False).to_string())

## 3.2 &nbsp; Confound test

Kruskal-Wallis tests whether distributions differ across groups without assuming
normality, which these skewed ratios violate.

In [ ]:
print('Kruskal-Wallis tests')
print('-' * 66)

for metric in ['days_cash', 'pt_svc_margin']:
    for grouping in ['control', 'hcris_class']:
        groups = [g[metric].dropna().values
                  for _, g in latest.groupby(grouping)
                  if g[metric].notna().sum() >= 3]
        h, p = kruskal(*groups)

        if p < 0.01:
            verdict = 'CONFOUNDED'
        elif p < 0.10:
            verdict = 'borderline'
        else:
            verdict = 'clear'

        print(f'{metric:16s} ~ {grouping:12s}   H = {h:6.2f}   p = {p:.4f}   {verdict}')

### Result

**Days cash on hand differs by ownership** (p = 0.0001) and not by Medicare class
(p = 0.22). Government hospital districts hold their own cash at a median near 120 days.
Proprietary corporations sweep it to a parent and report a median near 8 days, often
with negative equity. That gap measures a treasury arrangement.

**Patient services margin is borderline on ownership** (p = 0.05) and clear on Medicare
class, so it holds as a direct cross-hospital comparison.

Liquidity enters the model as a within-stratum percentile rank. Operating margin enters
as a raw value.

In [ ]:
for col in ['days_cash', 'equity_ratio']:
    panel[f'{col}_pct'] = panel.groupby(['fy', 'stratum'])[col].rank(pct=True)

print('Raw value vs within-stratum rank')
print(panel[['hospital', 'fy', 'stratum', 'days_cash', 'days_cash_pct']]
      .head(8).round(3).to_string(index=False))

## 3.3 &nbsp; Direction of travel

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))

for _, g in panel.groupby('ccn'):
    if g['days_cash'].notna().sum() >= 2:
        ax.plot(g['fy'], g['days_cash'], color='gray', alpha=0.25, linewidth=0.8)

median = panel.groupby('fy')['days_cash'].median()
ax.plot(median.index, median.values, color='#1F3864', linewidth=2.5,
        marker='o', label='Median')

ax.set_ylim(0, 400)
ax.set_xticks(sorted(panel['fy'].unique()))
ax.set_ylabel('Days cash on hand')
ax.set_title('Liquidity trajectory, one line per hospital')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

trend = panel.groupby('fy').agg(
    n=('ccn', 'size'),
    days_cash=('days_cash', 'median'),
    pt_svc_margin=('pt_svc_margin', 'median'),
    total_margin=('total_margin', 'median'),
    under_60_days=('days_cash', lambda s: int((s < 60).sum())),
    negative_margin=('total_margin', lambda s: int((s < 0).sum())),
).round(3)

print(trend.to_string())

## 3.4 &nbsp; Hospital-level features

One row per hospital, with three views of each metric: **level**, **slope** and
**volatility**. This distinguishes a hospital that is stable and weak from one that is
deteriorating.

In [ ]:
def slope(group, column):
    """Least-squares slope across available years. NaN if fewer than 2 points."""
    s = group.dropna(subset=[column])
    if len(s) < 2:
        return np.nan
    return linregress(s['fy'], s[column]).slope


METRICS = ['days_cash_pct', 'equity_ratio_pct', 'pt_svc_margin',
           'total_margin', 'days_cash', 'npr', 'beds', 'fte']

rows = []
for ccn, g in panel.groupby('ccn'):
    g = g.sort_values('fy')
    last = g.iloc[-1]

    row = {
        'ccn':         ccn,
        'hospital':    last['hospital'],
        'stratum':     last['stratum'],
        'control':     last['control'],
        'hcris_class': last['hcris_class'],
        'n_years':     len(g),
    }
    for m in METRICS:
        row[f'{m}_lvl']   = last[m]
        row[f'{m}_slope'] = slope(g, m)
        row[f'{m}_vol']   = g[m].std() if g[m].notna().sum() >= 2 else np.nan

    rows.append(row)

feat = pd.DataFrame(rows)
print(f'{len(feat)} hospitals, {feat.shape[1]} engineered columns')
feat.head()

## 3.5 &nbsp; Feature selection

With 64 hospitals the model carries six features, one from each block: liquidity,
capital structure, profitability, scale. Extreme values are winsorized at the 5th and
95th percentiles, since the outliers here are real hospitals.

In [ ]:
FEATURES = [
    'days_cash_pct_lvl',     # liquidity, ranked within ownership stratum
    'equity_ratio_pct_lvl',  # capital structure, ranked within stratum
    'pt_svc_margin_lvl',     # operating performance, where it stands
    'pt_svc_margin_slope',   # operating performance, where it is heading
    'total_margin_lvl',      # bottom line including non-patient revenue
    'npr_lvl',               # scale
]

X = feat[FEATURES].copy()
X['npr_lvl'] = np.log10(X['npr_lvl'].clip(lower=1))

complete = X.notna().all(axis=1)
Xc = X[complete].copy()

for c in Xc.columns:
    lo, hi = Xc[c].quantile([0.05, 0.95])
    Xc[c] = Xc[c].clip(lo, hi)

Xs = StandardScaler().fit_transform(Xc)
sub = feat[complete].reset_index(drop=True)

print(f'{complete.sum()} of {len(feat)} hospitals complete on all six features')
print()
print('Spearman correlation')
print(pd.DataFrame(Xs, columns=FEATURES).corr(method='spearman').round(2).to_string())

---
# 4 &nbsp; Clustering

## 4.1 &nbsp; Method

**Ward-linkage hierarchical clustering**, deterministic and requiring no commitment to
a number of groups in advance. K-means runs alongside as a cross-check.

**Bootstrap stability** decides how many groups are reported:

1. Resample hospitals with replacement and cluster the resample
2. Assign all original hospitals to the nearest resulting cluster centre
3. Measure Jaccard overlap between each original cluster and its best match
4. Repeat 400 times and average

Thresholds: **0.60 and above is stable**, **0.75 and above is highly stable**, below
0.50 indicates an artefact of the sample. A solution is reported only when every
cluster clears 0.60.

In [ ]:
def ward(X, k):
    """Ward-linkage flat clustering into k groups."""
    return fcluster(linkage(X, method='ward'), k, criterion='maxclust')


def clusterboot(X, k, n_boot=400, seed=0):
    """Bootstrap cluster stability. Returns base labels and mean Jaccard per cluster."""
    rng = np.random.default_rng(seed)
    base = ward(X, k)
    jaccard = np.zeros((n_boot, k))

    for b in range(n_boot):
        idx = rng.integers(0, len(X), len(X))
        Xb = X[idx]
        labels_b = ward(Xb, k)

        centroids = np.array([Xb[labels_b == c].mean(axis=0) for c in range(1, k + 1)])
        assigned = np.argmin(cdist(X, centroids), axis=1) + 1

        for c in range(1, k + 1):
            original = set(np.where(base == c)[0])
            best = 0.0
            for rc in range(1, k + 1):
                resampled = set(np.where(assigned == rc)[0])
                union = len(original | resampled)
                if union:
                    best = max(best, len(original & resampled) / union)
            jaccard[b, c - 1] = best

    return base, jaccard.mean(axis=0)

## 4.2 &nbsp; Number of groups

In [ ]:
print(f"{'k':>2}  {'silhouette':>10}  {'ARI vs kmeans':>13}   cluster sizes and stability")
print('-' * 90)

for k in range(2, 6):
    base, jac = clusterboot(Xs, k)
    km = KMeans(n_clusters=k, n_init=25, random_state=0).fit(Xs)
    sizes = np.bincount(base)[1:]

    detail = '   '.join(f'n={s:<3d} J={j:.2f}' for s, j in zip(sizes, jac))
    print(f'{k:>2}  {silhouette_score(Xs, base):>10.3f}  '
          f'{adjusted_rand_score(base, km.labels_):>13.2f}   '
          f'{detail}   mean J = {jac.mean():.3f}')

### Result

**Two groups**, stable at roughly 0.65 and 0.73.

From k = 3 upward a four-hospital cluster appears with a Jaccard near 0.38. At k = 4
only the largest cluster clears the threshold and mean stability falls to 0.53.
Silhouettes run 0.20 to 0.24 across the range, consistent with a single split rather
than a graded set of tiers.

In [ ]:
K = 2
sub['cluster'] = ward(Xs, K)

fig, ax = plt.subplots(figsize=(13, 4.5))
dendrogram(linkage(Xs, method='ward'),
           labels=sub['hospital'].values,
           leaf_rotation=90,
           leaf_font_size=6,
           ax=ax)
ax.set_title('Ward linkage, Louisiana rural hospitals')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

---
# 5 &nbsp; Interpretation

## 5.1 &nbsp; Profile of the two groups

In [ ]:
profile_cols = ['days_cash_pct_lvl', 'equity_ratio_pct_lvl', 'pt_svc_margin_lvl',
                'pt_svc_margin_slope', 'total_margin_lvl', 'days_cash_lvl']

print('Cluster profile, medians')
print(sub.groupby('cluster')[profile_cols].median().round(3).to_string())
print()
print('Sizes:', np.bincount(sub['cluster'])[1:])

| | Cluster 1 (n = 27) | Cluster 2 (n = 33) |
|---|---|---|
| Patient services margin | −30% | −8% |
| Direction of travel | worsening | improving |
| Total margin | −4% | +12% |
| Days cash on hand | 39 | 108 |
| Equity percentile within stratum | 0.33 | 0.69 |

Both groups lose money delivering care, which is the standard condition for a Louisiana
rural hospital. The split is the size of the gap and whether it is closing or widening.

## 5.2 &nbsp; What separates them

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(Xs, sub['cluster'])
print('Split rule')
print(export_text(tree, feature_names=FEATURES))

for col in ['hcris_class', 'stratum']:
    print(f'Cluster by {col}')
    print(pd.crosstab(sub['cluster'], sub[col]).to_string())
    print()

The split runs on total margin and equity rank, and spreads across both Medicare class
and ownership stratum. It is a new grouping, not a restatement of CAH status or of who
owns the hospital.

## 5.3 &nbsp; Testing the premise

The premise from section 1 was that financial position drives who joins LIHNC and who
engages with Epic. Predict membership in each from the financial features alone.
Accuracy at or below the base rate means financial position carries no information
about that grouping.

In [ ]:
flags = xwalk[['ccn', 'lihnc_member', 'in_epic_pipeline']]
sub = sub.merge(flags, on='ccn', how='left')

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

targets = {
    'CAH designation': (sub['hcris_class'] == 'CAH').astype(int),
    'LIHNC member':    (sub['lihnc_member'] == 'Yes').astype(int),
    'Epic pipeline':   (sub['in_epic_pipeline'] == 'Yes').astype(int),
}

print(f"{'Grouping':<18}{'base rate':>11}{'CV accuracy':>13}{'lift':>8}   verdict")
print('-' * 72)

for name, y in targets.items():
    base_rate = max(y.mean(), 1 - y.mean())
    acc = cross_val_score(LogisticRegression(C=0.5, max_iter=2000),
                          Xs, y, cv=cv, scoring='accuracy').mean()
    lift = acc - base_rate
    verdict = 'carries signal' if lift > 0.05 else 'no financial signal'
    print(f'{name:<18}{base_rate:>11.2f}{acc:>13.2f}{lift:>+8.2f}   {verdict}')

### Result: the premise fails

**CAH designation is predictable** from financial position, as reimbursement class and
finances are linked.

**LIHNC membership is not. Epic pipeline participation is not.** Both score below the
base rate, so the financial features perform worse than guessing the majority class.

Financial position played no part in determining who joined the network or who entered
the Epic waves. Neither grouping works as a proxy for readiness, and if either program
is meant to reach hospitals under financial pressure, it is not currently selecting
for that.

The question becomes what does explain the grouping.

## 5.4 &nbsp; Scoring VBC potential

Two of the three requirements from section 1 are measurable from cost report data. The
third is measurable from no available dataset and is carried as a blank.

| Axis | Source | Status |
|---|---|---|
| **Risk-bearing capacity** | Liquidity and equity, ranked within ownership stratum | Scored |
| **Operating performance** | Patient services margin, its trend, and total margin | Scored |
| **Attribution capacity** | Primary care providers billing under the hospital's own TIN, and certified provider-based RHCs | **No data. Requires a survey** |

The axes stay separate. A hospital strong on both scored axes and empty on the third is
not a candidate, and a combined score would conceal that.

In [ ]:
z = pd.DataFrame(Xs, columns=FEATURES)

sub['risk_capacity']  = z[['days_cash_pct_lvl', 'equity_ratio_pct_lvl']].mean(axis=1)
sub['operating_perf'] = z[['pt_svc_margin_lvl', 'pt_svc_margin_slope',
                           'total_margin_lvl']].mean(axis=1)
sub['attribution']    = np.nan   # awaiting the primary care and billing TIN survey

for c in ['risk_capacity', 'operating_perf']:
    sub[f'{c}_pct'] = sub[c].rank(pct=True).round(2)

fig, ax = plt.subplots(figsize=(7.5, 6))

palette = {1: ('#E24B4A', 'Cluster 1  (n=27)'),
           2: ('#1D9E75', 'Cluster 2  (n=33)')}

for cl, (colour, label) in palette.items():
    s = sub[sub['cluster'] == cl]
    ax.scatter(s['operating_perf'], s['risk_capacity'], c=colour, s=48,
               alpha=0.75, edgecolor='white', linewidth=0.5, label=label)

ax.axhline(0, color='#c3c2b7', linewidth=0.8)
ax.axvline(0, color='#c3c2b7', linewidth=0.8)
ax.set_xlabel('Operating performance  (standardized)')
ax.set_ylabel('Risk-bearing capacity  (within ownership stratum)')
ax.set_title('Two scored axes. The third, attribution, has no data.')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
scorecard = (sub[['hospital', 'hcris_class', 'stratum', 'cluster',
                  'risk_capacity_pct', 'operating_perf_pct', 'attribution',
                  'days_cash_lvl', 'pt_svc_margin_lvl', 'total_margin_lvl']]
             .sort_values('operating_perf_pct', ascending=False)
             .round(3))

scorecard.to_csv('la_hospital_scorecard.csv', index=False)
print(f'Wrote la_hospital_scorecard.csv  ({len(scorecard)} hospitals)')

# In Colab: from google.colab import files; files.download('la_hospital_scorecard.csv')

scorecard.head(15)

---
# 6 &nbsp; What explains the grouping

## 6.1 &nbsp; LIHNC and Epic track each other

Financial data does not explain either grouping. Before looking further, test whether
the two groupings explain each other.

In [ ]:
sub['lihnc'] = (sub['lihnc_member'] == 'Yes').astype(int)
sub['epic']  = (sub['in_epic_pipeline'] == 'Yes').astype(int)

from scipy.stats import fisher_exact

overlap = pd.crosstab(sub['lihnc'], sub['epic'])
overlap.index = ['Not LIHNC', 'LIHNC']
overlap.columns = ['Not in Epic pipeline', 'In Epic pipeline']

odds, p = fisher_exact(overlap.values)

print(overlap.to_string())
print()
print(f'Fisher exact test:  odds ratio = {odds:.2f},  p = {p:.4f}')

### Result

The two groupings are **strongly associated with each other** while neither is
associated with financial position.

A LIHNC member is roughly five to six times more likely to be in the Epic pipeline than
a non-member. Whatever produced the LIHNC roster also produced the Epic pipeline, and it
is not financial.

That is consistent with a relationship-driven mechanism: shared referral patterns, the
same regional hospital association meetings, CEOs who already knew each other, one
hospital recruiting its neighbours. None of those appear in a cost report.

## 6.2 &nbsp; Screening the non-financial variables

Test everything else available for the full universe against LIHNC membership. Size,
staffing, payer mix and market trajectory.

In [ ]:
from scipy.stats import mannwhitneyu

candidates = {
    'Beds':                'beds_lvl',
    'FTE employees':       'fte_lvl',
    'Net patient revenue': 'npr_lvl',
    'Days cash on hand':   'days_cash_lvl',
}

print(f"{'Variable':<36}{'LIHNC median':>16}{'Non-member':>16}{'p':>9}")
print('-' * 78)

for label, v in candidates.items():
    if v not in sub.columns:
        print(f'{label:<36}{"not in feature table":>41}')
        continue
    a = sub.loc[sub['lihnc'] == 1, v].dropna()
    b = sub.loc[sub['lihnc'] == 0, v].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = mannwhitneyu(a, b)
    print(f'{label:<36}{a.median():>16,.1f}{b.median():>16,.1f}{p:>9.3f}')

print()
print('LIHNC membership by Medicare class')
print(pd.crosstab(sub['hcris_class'], sub['lihnc_member']).to_string())

### Result

Bed count, staffing and revenue come back null. **Raw days cash on hand does not**:
LIHNC members hold a median of 98 days against 22 for non-members (p = 0.015).

That appears to contradict section 5.3, where financial features could not predict
membership. The next cell resolves it. The model used liquidity *ranked within
ownership stratum*, and the raw difference is a composition effect.

## 6.3 &nbsp; LIHNC is a network of public hospital districts

In [ ]:
from scipy.stats import fisher_exact

composition = pd.crosstab(sub['stratum'], sub['lihnc_member'])
print('Ownership stratum by LIHNC membership')
print(composition.to_string())

gov_prop = composition.loc[['Government', 'Proprietary']]
_, p_comp = fisher_exact(gov_prop.values)
print(f'\nGovernment vs Proprietary composition, Fisher p = {p_comp:.4f}')

print('\nLiquidity, raw against stratum-adjusted')
for col, label in [('days_cash_lvl', 'Days cash, raw'),
                   ('days_cash_pct_lvl', 'Days cash, within-stratum rank')]:
    a = sub.loc[sub['lihnc_member'] == 'Yes', col].dropna()
    b = sub.loc[sub['lihnc_member'] == 'No',  col].dropna()
    _, p = mannwhitneyu(a, b)
    print(f'  {label:<32}LIHNC = {a.median():>8.2f}   non = {b.median():>8.2f}   p = {p:.3f}')

gov = sub[sub['stratum'] == 'Government']
a = gov.loc[gov['lihnc_member'] == 'Yes', 'days_cash_lvl'].dropna()
b = gov.loc[gov['lihnc_member'] == 'No',  'days_cash_lvl'].dropna()
_, p_gov = mannwhitneyu(a, b)
print(f'\nWithin the Government stratum only (n = {len(a)} vs {len(b)}): '
      f'LIHNC = {a.median():.1f}, non = {b.median():.1f}, p = {p_gov:.3f}')

### Result: ownership is the organizing fact

**21 of 24 LIHNC members are government hospital districts.** One is proprietary,
against 12 proprietary hospitals outside the network (Fisher p = 0.003).

The raw liquidity gap disappears entirely once ownership is accounted for. Within-stratum
rank gives p = 0.95, and comparing only government hospitals to each other gives p = 0.97.
LIHNC members look more liquid because districts hold their own cash while corporate
hospitals sweep it to a parent. The network did not recruit hospitals with money. It
recruited public entities.

That is consistent with everything else here. Public hospital districts share governance
structures, appear before the same parish boards, and have executives who meet through
the same public-sector channels. Membership followed those relationships, not a balance
sheet.

It also identifies the unrecruited population precisely: **the 12 proprietary hospitals
outside LIHNC**, whose participation is decided by a corporate parent rather than a local
board, and who are the financially weakest stratum in the state.

### Result

Nothing in the available data separates members from non-members. Bed count, staffing,
revenue and Medicare class all come back non-significant. Parish Medicaid enrollment
change is the closest, and it does not clear conventional thresholds.

The grouping is not explained by any structural or financial characteristic that a cost
report records. This points at geography and relationships, neither of which is in the
data as it stands.

## 6.4 &nbsp; Who gains most from a shared EHR instance

A separate question from network membership, and one the pipeline data can answer.

Epic implementation is priced largely per provider seat, so the same instance imposes
very different burdens depending on a hospital's size and how much volume it runs
through each provider. Three measures:

- **Cost as a share of net patient revenue** — the affordability question
- **Cost per provider** — whether the hospital is getting standard per-seat pricing
- **Cost per encounter** — how much clinical volume each licensed seat carries

A hospital with high volume per provider extracts more value from each seat. A hospital
paying a large share of its revenue cannot fund the build alone, which is precisely
where a shared instance and subsidy do the most work.

---
# 7 &nbsp; Does location explain the grouping?

Financial position explains neither program. The Louisiana Health Atlas provides
ZIP-level population, rurality, social vulnerability and facility counts, which lets
the geographic hypothesis be tested directly.

## 7.1 &nbsp; Join hospitals to their ZIP

Hospital ZIP codes come from the cost report. Five hospitals file under a PO Box ZIP,
which has no atlas record, so those are remapped to the physical ZIP of the same city.

In [ ]:
ATLAS = BASE + 'louisiana_health_atlas_export_zip.csv'

atlas = pd.read_table(ATLAS, sep=',', header=0, low_memory=False)
atlas['zip'] = atlas['ZIP Code'].astype(str).str.zfill(5)
atlas = atlas.drop_duplicates('zip')

# Hospital ZIP from the most recent cost report
zips = pd.read_table(FILES[2023], sep=',', header=0, low_memory=False)
zips = zips[zips['State Code'] == 'LA'].copy()
zips['ccn'] = zips['Provider CCN'].astype(str).str.zfill(6)
zips['zip'] = zips['Zip Code'].astype(str).str.extract(r'(\d{5})')[0]

# PO Box ZIPs have no atlas record. Remap to the physical ZIP of the same city.
PO_BOX = {'70511': '70510',   # Abbeville
          '70562': '70560',   # New Iberia
          '71121': '71220',   # Bastrop
          '70157': '70517',   # Breaux Bridge
          '70308': '70380'}   # Morgan City
zips['zip'] = zips['zip'].replace(PO_BOX)

place = (sub[['ccn', 'hospital', 'cluster', 'lihnc', 'epic']]
         .merge(zips[['ccn', 'zip']], on='ccn', how='left')
         .merge(atlas, on='zip', how='left'))

place['ruca'] = pd.to_numeric(place['Rural Designation (RUCA Category)'], errors='coerce')

print(f'{place["Population"].notna().sum()} of {len(place)} hospitals matched to a ZIP record')
print()
print(place['Classification'].value_counts().to_string())

## 7.2 &nbsp; Place characteristics against each program

RUCA runs 1 to 10. Low values are metropolitan or metro-adjacent, high values are small
town and isolated. Mann-Whitney compares members against non-members on each atlas
variable.

In [ ]:
from scipy.stats import mannwhitneyu

ATLAS_VARS = {
    'Population':                          'ZIP population',
    'ruca':                                'RUCA category',
    'Social Vulnerability (Poverty)':      'Poverty',
    'Social Vulnerability (Food Access)':  'Food access',
    'Transportation (Vehicle Ownership)':  'No vehicle',
    'Diabetes Prevalence':                 'Diabetes prevalence',
    'Broadband Deserts':                   'Broadband desert',
    'Healthcare Facility (Acute)':         'Acute facilities in ZIP',
    'Healthcare Facility (Specialty)':     'Specialty facilities in ZIP',
}

for target, name in [('lihnc', 'LIHNC membership'), ('epic', 'Epic pipeline')]:
    print(f'{name}')
    print(f"{'Variable':<30}{'member':>12}{'non-member':>13}{'p':>9}")
    print('-' * 66)
    for col, label in ATLAS_VARS.items():
        a = place.loc[place[target] == 1, col].dropna()
        b = place.loc[place[target] == 0, col].dropna()
        if len(a) < 5 or len(b) < 5:
            continue
        _, p = mannwhitneyu(a, b)
        mark = '  <<<' if p < 0.05 else ''
        print(f'{label:<30}{a.median():>12,.1f}{b.median():>13,.1f}{p:>9.3f}{mark}')
    print()

### Result

For **LIHNC**, nothing is significant. Population, poverty, food access, vehicle
ownership, diabetes prevalence, broadband and facility counts all come back null.
Community need did not sort hospitals into the network.

For **Epic**, RUCA category separates the groups (p = 0.03). Pipeline hospitals sit at a
median RUCA of 7 against 4 for non-participants. The Epic pipeline reaches further into
isolated Louisiana than the rest of the universe.

## 7.3 &nbsp; Rurality, with the two programs controlling for each other

The two rosters overlap heavily, so a univariate test cannot tell whether rurality acts
on Epic, on LIHNC, or on whatever they share. Fit each program on RUCA with the other
program included.

In [ ]:
import statsmodels.api as sm

model_data = place[['ruca', 'lihnc', 'epic']].dropna().astype(float)

print('Epic pipeline ~ RUCA + LIHNC')
X = sm.add_constant(model_data[['ruca', 'lihnc']])
print(sm.Logit(model_data['epic'], X).fit(disp=0).summary2().tables[1].round(3).to_string())

print()
print('LIHNC membership ~ RUCA + Epic')
X = sm.add_constant(model_data[['ruca', 'epic']])
print(sm.Logit(model_data['lihnc'], X).fit(disp=0).summary2().tables[1].round(3).to_string())

print()
print('Participation rate by RUCA band')
place['band'] = pd.cut(place['ruca'], [0, 3, 6, 10],
                       labels=['1-3 metro / adjacent',
                               '4-6 micropolitan',
                               '7-10 small town / isolated'])
print(place.groupby('band', observed=True)
      .agg(n=('epic', 'size'), epic_rate=('epic', 'mean'), lihnc_rate=('lihnc', 'mean'))
      .round(2).to_string())

### Result: the two programs load on rurality in opposite directions

| Program | RUCA coefficient | p |
|---|---|---|
| Epic pipeline | **+0.25** | 0.010 |
| LIHNC membership | **−0.20** | 0.041 |

Epic participation rises with isolation. LIHNC membership falls with it. Each effect
holds with the other program in the model, so this is not the shared driver showing up
twice.

| Location | n | Epic rate | LIHNC rate |
|---|---|---|---|
| Metro or metro-adjacent (RUCA 1–3) | 18 | 17% | **56%** |
| Micropolitan (4–6) | 13 | 38% | 31% |
| Small town or isolated (7–10) | 33 | **45%** | 30% |

Only 1 of 12 hospitals in an urban ZIP is in the Epic pipeline. LIHNC is a
metro-adjacent network; the Epic pipeline reaches outward.

## 7.4 &nbsp; Where the financial clusters sit

Test the place variables against the financial clustering from section 4.

In [ ]:
print(f"{'Variable':<30}{'Cluster 1':>12}{'Cluster 2':>12}{'p':>9}")
print('-' * 65)

for col, label in ATLAS_VARS.items():
    a = place.loc[place['cluster'] == 1, col].dropna()
    b = place.loc[place['cluster'] == 2, col].dropna()
    if len(a) < 5 or len(b) < 5:
        continue
    _, p = mannwhitneyu(a, b)
    mark = '  <<<' if p < 0.05 else ''
    print(f'{label:<30}{a.median():>12,.1f}{b.median():>12,.1f}{p:>9.3f}{mark}')

print()
print('Share of hospitals in the weaker cluster, by ZIP population tercile')
place['pop_band'] = pd.qcut(place['Population'], 3, labels=['small', 'mid', 'large'])
print(place.groupby('pop_band', observed=True)
      .agg(n=('cluster', 'size'),
           share_weak=('cluster', lambda s: (s == 1).mean()),
           median_ruca=('ruca', 'median'))
      .round(2).to_string())

### Result: the financially weaker hospitals are in the larger towns

Cluster 1, the weaker group, sits in ZIPs with a median population of about 15,800.
Cluster 2 sits at about 8,800 (p = 0.05). Seventy percent of hospitals in the largest
population tercile fall in the weaker cluster, against 30% in the smallest.

The mechanism looks like competition rather than size. **Specialty facility count in the
hospital's own ZIP is 0.85 for the weaker cluster and 0.33 for the stronger one**
(p = 0.005), while acute facility count shows no difference. It is not other hospitals.
It is ambulatory surgery, imaging and specialty clinics taking the profitable outpatient
volume.

Isolated hospitals hold a captive market, and the Critical Access Hospitals among them
add cost-based reimbursement on top. That combination is doing more for their balance
sheets than management is.

## 7.5 &nbsp; What this means for hub design

Two networks are organized on incompatible geography.

**LIHNC clusters near metro areas**, where hospitals face outpatient competition and are
financially weaker. **The Epic pipeline reaches outward**, to hospitals that are
financially steadier but have the least infrastructure and the fewest neighbours.

The needs invert the same way. Isolated hospitals need connectivity, which is what a
shared EHR instance delivers. Metro-adjacent hospitals need volume defence against
specialty competition, which is what network contracting and value-based arrangements
deliver. Each group is currently enrolled in more of the other one's programme.

Hub design has to reconcile that rather than assume the two rosters describe the same
set of relationships. Two inputs remain outstanding:

| Input | Source | Answers |
|---|---|---|
| **Drive times between hospitals** | Geocode cost report addresses, compute an origin-destination matrix | Which hospitals are close enough to share staff, call coverage and a service desk |
| **Referral and outmigration flows** | LDH inpatient discharge data, Bureau of Health Informatics | Which small hospitals already send patients to which larger ones |

The constraint the financial data already imposes: the three rural referral centres are
the natural hub anchors on referral logic and hold a median of 1.4 days cash. Any hub
design has to resolve that before it can be proposed.

In [ ]:
# Epic pipeline economics. Requires the pipeline columns from the crosswalk.
epic_cols = ['epic_est_pricing_m', 'epic_amb_vol', 'epic_ip_vol', 'epic_providers']
have_epic = all(c in xwalk.columns for c in epic_cols)

if not have_epic:
    print('Epic pipeline columns not present in rural_ccn.csv.')
    print('Add: epic_wave, epic_est_pricing_m, epic_amb_vol, epic_ip_vol, epic_providers')
else:
    ep = xwalk[['ccn', 'epic_wave'] + epic_cols].copy()
    for c in epic_cols:
        ep[c] = pd.to_numeric(ep[c], errors='coerce')

    ep = ep.merge(xwalk[['ccn', 'hospital']], on='ccn', how='left')
    ep = ep.merge(sub[['ccn', 'days_cash_lvl', 'npr_lvl', 'cluster']], on='ccn')
    ep = ep.dropna(subset=['epic_est_pricing_m', 'epic_providers'])

    encounters = ep['epic_amb_vol'].fillna(0) + ep['epic_ip_vol'].fillna(0)
    cost = ep['epic_est_pricing_m'] * 1e6

    ep['cost_pct_npr']      = (cost / ep['npr_lvl']).round(3)
    ep['cost_per_provider'] = (cost / ep['epic_providers']).round(0)
    ep['cost_per_encounter']= (cost / encounters.replace(0, np.nan)).round(2)

    show = ep[['hospital', 'epic_wave', 'epic_est_pricing_m', 'epic_providers',
               'cost_pct_npr', 'cost_per_provider', 'cost_per_encounter',
               'days_cash_lvl']].sort_values('cost_pct_npr')

    print(show.to_string(index=False))
    print()
    print('Median cost per provider: $' + f"{ep['cost_per_provider'].median():,.0f}")
    print(f"Median cost as share of net patient revenue: {ep['cost_pct_npr'].median():.1%}")